In [5]:
import pandas as pd
import numpy as np
from scipy import stats

In [6]:
df_data = pd.read_excel("/Volumes/T7 1TB SSD/Empirical_Analysis_FM/Applied_Research_Framework_Git/Applied_projects/Crude_oil_market_analysis/Regression_Urals_loading_analysis_mod.xlsx", sheet_name="Data")
print(df_data)

          Date  LCOc1  NWEMURLCRKMc1   .IMOEX    RSX  BFO-URL-NWE  \
0   2017-01-06  56.75      -0.545276  2213.93  21.52        -2.50   
1   2017-01-13  55.59       1.238238  2195.19  21.45        -2.05   
2   2017-01-20  55.49       0.386498  2159.96  21.08        -1.50   
3   2017-01-27  55.46       0.348593  2266.05  22.08        -1.60   
4   2017-02-03  56.72       0.219638  2226.61  21.80        -1.75   
..         ...    ...            ...      ...    ...          ...   
230 2021-10-22  85.77       7.090000  4196.96  32.67        -1.80   
231 2021-10-29  84.38       3.930000  4150.00  32.00        -1.40   
232 2021-11-05  82.55       1.860000  4174.76  31.93        -1.60   
233 2021-11-12  81.95       1.750000  4121.66  31.27        -1.45   
234 2021-11-19  78.66       0.030000  4016.47  29.92        -1.65   

     Urals loading  
0     1.391864e+07  
1     1.480585e+07  
2     1.377569e+07  
3     1.332605e+07  
4     1.538791e+07  
..             ...  
230   1.347257e+07  
231

In [ ]:
# complete correlation structure analysis including Urals loading
merged_df = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX','LCOc1']]

print(merged_df)

df_data_standardized = (merged_df - merged_df.mean()) / merged_df.std()
df_data_zscore = df_data_standardized[(np.abs(stats.zscore(df_data_standardized)) < 3).all(axis=1)]

corr_matrix_merged = df_data_zscore[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX','LCOc1']].corr()

     Urals loading  NWEMURLCRKMc1   .IMOEX  LCOc1
0     1.391864e+07      -0.545276  2213.93  56.75
1     1.480585e+07       1.238238  2195.19  55.59
2     1.377569e+07       0.386498  2159.96  55.49
3     1.332605e+07       0.348593  2266.05  55.46
4     1.538791e+07       0.219638  2226.61  56.72
..             ...            ...      ...    ...
230   1.347257e+07       7.090000  4196.96  85.77
231   1.275017e+07       3.930000  4150.00  84.38
232   9.529337e+06       1.860000  4174.76  82.55
233   3.400765e+06       1.750000  4121.66  81.95
234   1.018841e+07       0.030000  4016.47  78.66

[235 rows x 4 columns]


In [8]:
corr_4x4_actual = np.array([
    [1.00,    -0.024,  -0.409,  0.229],    # Urals loading 
    [-0.024,  1.00,     0.219,  0.221],    # NWEMURLCRKMc1   
    [-0.409,  0.219,    1.00,   0.292],    # .IMOEX    
    [0.229,   0.221,    0.292,  1.00]      # LCOc1
    ])

In [9]:

means = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX', 'LCOc1']].mean().values
stds = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX', 'LCOc1']].std().values

In [10]:
cov_4x4 = np.outer(stds, stds) * corr_4x4_actual

In [11]:
# generate synthetic
n_obs = 234
np.random.seed(42)
synthetic_data = np.random.multivariate_normal(
    mean=means,
    cov=cov_4x4,
    size=234)

print(f"✓ generated successfully: shape {synthetic_data.shape}")

✓ generated successfully: shape (234, 4)


In [ ]:
# verification synthetic has same correlations as real

df_synthetic = pd.DataFrame(synthetic_data,
columns=['Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1'])
print("real correlations:")
print(corr_matrix_merged)

print("\nsynthetic correlations:")
print(df_synthetic.corr())

real correlations:
               Urals loading  NWEMURLCRKMc1    .IMOEX     LCOc1
Urals loading       1.000000      -0.023755 -0.409401  0.229433
NWEMURLCRKMc1      -0.023755       1.000000  0.218897  0.221110
.IMOEX             -0.409401       0.218897  1.000000  0.291510
LCOc1               0.229433       0.221110  0.291510  1.000000

synthetic correlations:
               Urals loading  NWEMURL.CRMc1    .IMOEX     LCOc1
Urals loading       1.000000       0.023830 -0.423747  0.227258
NWEMURL.CRMc1       0.023830       1.000000  0.243727  0.244638
.IMOEX             -0.423747       0.243727  1.000000  0.228545
LCOc1               0.227258       0.244638  0.228545  1.000000


In [15]:
start_date = pd.Timestamp('2017-01-06')
df_synthetic['date'] = pd.date_range(start=start_date, periods=n_obs, freq='W')

# reorder columns: date first, then data
df_synthetic = df_synthetic[['date','Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1']]

In [14]:
df_synthetic.to_csv('urals_synthetic_data.csv', index=False)

print(f"\nfirst 10 rows of synthetic data:")
print(df_synthetic.head(10))
print(f"\ndata shape: {df_synthetic.shape}")
print(f"columns: {list(df_synthetic.columns)}")


first 10 rows of synthetic data:
        date  Urals loading  NWEMURL.CRMc1       .IMOEX      LCOc1
0 2017-01-08   1.059703e+07      -2.437084  2919.842865  67.715078
1 2017-01-15   1.284399e+07       0.482367  2789.741086  80.438358
2 2017-01-22   1.356745e+07       2.705914  2294.991972  54.363780
3 2017-01-29   1.138023e+07       4.263067  3851.931309  51.458729
4 2017-02-05   1.523793e+07       5.612465  2286.495648  52.100909
5 2017-02-12   7.618168e+06       6.782592  3212.656670  58.566130
6 2017-02-19   1.379775e+07      -0.019098  2518.407980  49.441688
7 2017-02-26   1.397070e+07      -4.125038  2730.104236  58.000781
8 2017-03-05   1.216561e+07       7.052637  3307.386982  75.546454
9 2017-03-12   1.148199e+07       2.090782  3869.596600  56.276376

data shape: (234, 5)
columns: ['date', 'Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1']
